# Extract and Transform 2025 NYC Yellow Taxi Data

In [2]:
import polars as pl

## Extract

In [3]:
def extract():
    return pl.scan_parquet("*.parquet")

## Transform

In [14]:
def update_payment_type(df):
    return df.with_columns(replaced=pl.col("payment_type").replace_strict([0, 1, 2, 3, 4, 5, 6], 
        ["FlexFareTrip", "CreditCard", "Cash", "NoCharge", "Dispute","Unknown","VoidedTrip"])
        ).drop("payment_type"
        ).rename({"replaced": "payment_type"})

def update_rate_code(df):
    return df.with_columns(replaced=pl.col("RatecodeID").replace_strict([1, 2, 3, 4, 5, 6, 99], 
    ["StandardRate", "JFK", "Newark", "NassauOrWestchester", "NegotiatedFare", "GroupRide", "NullUnknown"])
        ).drop("RatecodeID"
        ).rename({"replaced": "RateCode"})

def drop_null_values(df):
    return df.drop_nulls()

def drop_unwanted_pickup_dates(df):
    df = df.with_columns(pl.col("tpep_pickup_datetime").dt.ordinal_day().alias("pickup_doy"))
    df = df.filter(~pl.col("pickup_doy").is_in([1, 244]))
    return df.drop("pickup_doy")

def drop_unwanted_dropoff_dates(df):
    df = df.with_columns(pl.col("tpep_dropoff_datetime").dt.ordinal_day().alias("dropoff_doy"))
    df = df.filter(~pl.col("dropoff_doy").is_in([151, 244, 245]))
    return df.drop("dropoff_doy")

def drop_negative_fares(df):
    return df.filter(pl.col("fare_amount") > 0)

# Load

In [12]:
def write_to_csv(df):
    df.sink_csv("2025_yellow_cab_data.csv")

# Pipeline

In [ ]:
# Extract
df = extract()

# Transform
df = update_payment_type(df)
df = update_rate_code(df)
df = drop_null_values(df)
df = drop_unwanted_pickup_dates(df)
df = drop_unwanted_dropoff_dates(df)
df = drop_negative_fares(df)

# Load
write_to_csv(df)